# DNAformer Gradient Attribution Analysis

**Purpose:** Run the same effective context radius analysis on DNAformer (with official weights)
to compare with DNA-GRU. If DNAformer's transformer also concentrates influence locally despite
having global attention capacity, it proves the *task* is local, not just the model.

**Requirements:**
- DNAformer weights at `DNA_weights/checkpoints/DNAFormer_siamese.pth`
- Real test data in `../Data/`
- `einops` package (`pip install einops`)

**Note:** The released weights are for the Nanopore/Illumina configuration (L=128, T=132).
Only `BinnedNanoporeTwoFlowcells` and `BinnedTestIllumina` can be evaluated.

## Imports

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from pathlib import Path
import os, random, json, math, time
from tqdm.notebook import tqdm
from collections import defaultdict
import scipy.io as sio
from einops import rearrange

os.environ["CUDA_VISIBLE_DEVICES"] = "2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE=="cuda" else ""))


Device: cuda (NVIDIA GeForce RTX 3080)


## Configuration

In [2]:
DATASET_NAME = "BinnedNanoporeTwoFlowcells"  # or "BinnedTestIllumina"

# DNAformer config -- EXACT values from their train.py / inference.py
class dnaformer_config:
    filter_index    = True
    index_length    = 12
    label_length    = 140       # full strand length
    corrupt_max_deviation = 4
    noisy_copies_length = label_length + corrupt_max_deviation  # 144
    if filter_index:
        noisy_copies_length -= index_length  # 132
    max_number_per_cluster = 16
    n_head             = 32
    activation         = 'gelu'
    num_layers         = 12
    d_model            = 1024
    alignement_filters = 128    # note: their typo, 'alignement'
    alignment_filters  = 128    # alias
    dim_feedforward    = 2048
    output_ch          = 4
    enc_filters        = 4      # one-hot DNA encoding channels
    p_dropout          = 0
    use_input_scaling  = False
    device             = DEVICE

# Derived
LABEL_SEQ_LEN = dnaformer_config.label_length - dnaformer_config.index_length  # 128
NOISY_LEN = dnaformer_config.noisy_copies_length  # 132
MAX_CLUSTER_SIZE = 16
BATCH_SIZE = 32  # smaller than DNA-GRU due to 103M params

# Paths
REAL_DATA_DIR = Path("../Data")
DATASET_FILES = {
    "BinnedTestIllumina": "BinnedTestIllumina.txt",
    "BinnedNanoporeTwoFlowcells": "BinnedNanoporeTwoFlowcells.txt",
}
EVAL_FILE = REAL_DATA_DIR / DATASET_FILES[DATASET_NAME]
DNAFORMER_WEIGHTS = Path("DNN_weights/checkpoints/DNAFormer_siamese.pth")

# Output -- in same experiment directory as DNA-GRU results
EXPERIMENT_DIR = Path(f"./Experiments/{DATASET_NAME}_v4")
THEORY_DIR = EXPERIMENT_DIR / "TheoryAnalysis"
T_RESULTS = THEORY_DIR / "Results"
T_MAT     = THEORY_DIR / "MatFiles"
T_FIGURES = THEORY_DIR / "Figures"
for d in [T_RESULTS, T_MAT, T_FIGURES]: d.mkdir(parents=True, exist_ok=True)

print(f"Dataset:  {DATASET_NAME} (L={LABEL_SEQ_LEN}, T={NOISY_LEN})")
print(f"Weights:  {DNAFORMER_WEIGHTS}")
print(f"Data:     {EVAL_FILE}")
print(f"Output:   {THEORY_DIR}")
assert DNAFORMER_WEIGHTS.exists(), f"Weights not found: {DNAFORMER_WEIGHTS}"
assert EVAL_FILE.exists(), f"Data not found: {EVAL_FILE}"


Dataset:  BinnedNanoporeTwoFlowcells (L=128, T=132)
Weights:  DNN_weights/checkpoints/DNAFormer_siamese.pth
Data:     ../Data/BinnedNanoporeTwoFlowcells.txt
Output:   Experiments/BinnedNanoporeTwoFlowcells_v4/TheoryAnalysis


## DNAformer Model
Exact code from `model_DNAFormer_siamese.py` in their GitHub repository.

In [3]:
# -- Exact DNAformer architecture from their GitHub --

class depthwise_separable_conv_1d(nn.Module):
    def __init__(self, in_ch, out_ch, kernels_per_layer=1, kernel_size=3, stride=1, padding=0):
        super().__init__()
        self.depthwise = nn.Conv1d(in_ch, in_ch * kernels_per_layer, kernel_size=kernel_size,
                                   stride=stride, padding=padding, groups=in_ch)
        self.pointwise = nn.Conv1d(in_ch * kernels_per_layer, out_ch, kernel_size=1)
    def forward(self, x):
        return self.pointwise(self.depthwise(x))

class double_conv1D(nn.Module):
    def __init__(self, config, in_ch, out_ch, seq_len, padding=0, kernel_size=3, stride=1, output_padding=0):
        super().__init__()
        self.conv = nn.Sequential(
            depthwise_separable_conv_1d(in_ch, out_ch, kernels_per_layer=1, kernel_size=kernel_size, stride=stride, padding=padding),
            nn.LayerNorm(seq_len, elementwise_affine=True),
            nn.GELU(),
            nn.Dropout(config.p_dropout),
            depthwise_separable_conv_1d(out_ch, out_ch, kernels_per_layer=1, kernel_size=kernel_size, stride=stride, padding=padding),
            nn.LayerNorm(seq_len, elementwise_affine=True),
            nn.GELU(),
            nn.Dropout(config.p_dropout))
    def forward(self, x):
        return self.conv(x)

class linear_block(nn.Module):
    def __init__(self, config, input_len, output_len):
        super().__init__()
        self.fc_1   = nn.Linear(input_len, output_len)
        self.norm_1 = nn.LayerNorm(output_len, elementwise_affine=True)
        self.act_1  = nn.GELU()
        self.dout_1 = nn.Dropout(config.p_dropout)
        self.fc_2   = nn.Linear(output_len, output_len)
        self.norm_2 = nn.LayerNorm(output_len, elementwise_affine=True)
        self.act_2  = nn.GELU()
        self.dout_2 = nn.Dropout(config.p_dropout)
        self.fc_3   = nn.Linear(output_len, output_len)
    def forward(self, x):
        x = self.dout_1(self.act_1(self.norm_1(self.fc_1(x))))
        x = self.dout_2(self.act_2(self.norm_2(self.fc_2(x))))
        return self.fc_3(x)

class alignement_module(nn.Module):
    def __init__(self, config, in_ch, out_ch):
        super().__init__()
        self.config = config
        self.conv_block_1 = double_conv1D(config, in_ch, out_ch//4, seq_len=config.noisy_copies_length, kernel_size=1)
        self.conv_block_2 = double_conv1D(config, in_ch, out_ch//4, seq_len=config.noisy_copies_length, kernel_size=3, padding=1)
        self.conv_block_3 = double_conv1D(config, in_ch, out_ch//4, seq_len=config.noisy_copies_length, kernel_size=5, padding=2)
        self.conv_block_4 = double_conv1D(config, in_ch, out_ch//4, seq_len=config.noisy_copies_length, kernel_size=7, padding=3)
        self.linear_block = linear_block(config, input_len=config.noisy_copies_length, output_len=config.noisy_copies_length)
    def forward(self, x):
        batch, cluster, emb, seq = x.shape
        x = rearrange(x, 'b cluster emb seq -> (b cluster) emb seq')
        x = torch.cat([self.conv_block_1(x), self.conv_block_2(x), self.conv_block_3(x), self.conv_block_4(x)], dim=1)
        x = self.linear_block(x)
        return rearrange(x, '(b cluster) emb seq -> b cluster emb seq', b=batch, cluster=cluster)

class embedding_module(nn.Module):
    def __init__(self, config, in_ch, out_ch):
        super().__init__()
        self.config = config
        self.label_length = config.label_length
        if config.filter_index:
            self.label_length -= config.index_length
        self.conv_block_1 = double_conv1D(config, in_ch, out_ch//4, seq_len=config.noisy_copies_length, kernel_size=1)
        self.conv_block_2 = double_conv1D(config, in_ch, out_ch//4, seq_len=config.noisy_copies_length, kernel_size=3, padding=1)
        self.conv_block_3 = double_conv1D(config, in_ch, out_ch//4, seq_len=config.noisy_copies_length, kernel_size=5, padding=2)
        self.conv_block_4 = double_conv1D(config, in_ch, out_ch//4, seq_len=config.noisy_copies_length, kernel_size=7, padding=3)
        self.linear_block = linear_block(config, input_len=config.noisy_copies_length, output_len=self.label_length)
    def forward(self, x):
        x = torch.sum(x, dim=1)  # NCI
        x = torch.cat([self.conv_block_1(x), self.conv_block_2(x), self.conv_block_3(x), self.conv_block_4(x)], dim=1)
        return self.linear_block(x)

class output_module(nn.Module):
    def __init__(self, config, in_ch, out_ch):
        super().__init__()
        self.conv_1 = nn.Conv1d(in_ch, in_ch, 1)
        self.conv_2 = nn.Conv1d(in_ch, in_ch, 1)
        self.conv_3 = nn.Conv1d(in_ch, out_ch, 1)
    def forward(self, x):
        return self.conv_3(self.conv_2(self.conv_1(x)))

class fusion_module(nn.Module):
    def __init__(self, config):
        super().__init__()
        label_length = config.label_length
        if config.filter_index:
            label_length -= config.index_length
        self.pred_fusion_left  = nn.Parameter(torch.ones(label_length))
        self.pred_fusion_right = nn.Parameter(torch.ones(label_length))
        self.conv_1 = nn.Conv1d(config.output_ch, config.output_ch, 1)
        self.conv_2 = nn.Conv1d(config.output_ch, config.output_ch, 1)
        self.conv_3 = nn.Conv1d(config.output_ch, config.output_ch, 1)
    def forward(self, x):
        x_left  = x[:x.shape[0]//2, :, :]
        x_right = torch.flip(x[x.shape[0]//2:, :, :], dims=[-1])
        x = (x_left * self.pred_fusion_left + x_right * self.pred_fusion_right) / 2
        return self.conv_3(self.conv_2(self.conv_1(x))), x_left, x_right

class DNAFormerNet(nn.Module):
    # Complete DNAformer Siamese model -- exact from their GitHub.
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.alignement = alignement_module(config, config.enc_filters, config.alignement_filters)
        self.embedding = embedding_module(config, config.alignement_filters, config.d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=config.d_model, dim_feedforward=config.dim_feedforward,
            nhead=config.n_head, activation=config.activation, dropout=config.p_dropout)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=config.num_layers)
        self.output_module = output_module(config, config.d_model, config.output_ch)
        self.fusion = fusion_module(config)

    def forward(self, x):
        x = self.alignement(x)
        x = self.embedding(x)
        x = self.encoder(rearrange(x, 'b emb seq -> seq b emb'))
        x = self.output_module(rearrange(x, 'seq b emb -> b emb seq'))
        x, x_left, x_right = self.fusion(x)
        return {'pred': x, 'pred_left': x_left, 'pred_right': x_right}

print(f"DNAformer model class defined.")


DNAformer model class defined.


## Dataset (one-hot encoding for DNAformer)

In [4]:
from torch.utils.data import Dataset, DataLoader

DNA_MAP = {'A': 0, 'C': 1, 'G': 2, 'T': 3}

def one_hot_encode(seq, max_len):
    # Convert DNA string to one-hot tensor (4 x max_len), zero-padded at end.
    t = torch.zeros(4, max_len, dtype=torch.float32)
    for i, c in enumerate(seq[:max_len]):
        if c in DNA_MAP:
            t[DNA_MAP[c], i] = 1.0
    return t

class DNAformerDataset(Dataset):
    # Dataset that produces one-hot encoded clusters for DNAformer.
    def __init__(self, filepath, max_cluster_size, noisy_len, label_len):
        self.max_cluster_size = max_cluster_size
        self.noisy_len = noisy_len
        self.label_len = label_len
        self.labels, self.clusters = [], []
        with open(filepath, 'r') as f:
            content = f.read()
        for block in content.split('\n\n'):
            if not block.strip(): continue
            lines = [l.strip() for l in block.strip().split('\n')]
            if len(lines) < 3: continue
            label_seq, reads = lines[0], [r for r in lines[2:] if r]
            if reads and label_seq:
                self.labels.append(label_seq)
                self.clusters.append(reads)
        print(f"Loaded {len(self.labels)} clusters from {filepath}")

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        reads = list(self.clusters[idx])
        random.shuffle(reads)
        # One-hot encode label (4 x label_len)
        label_oh = one_hot_encode(self.labels[idx], self.label_len)
        # One-hot encode reads into (K, 4, noisy_len)
        cluster_left  = torch.zeros(self.max_cluster_size, 4, self.noisy_len)
        cluster_right = torch.zeros(self.max_cluster_size, 4, self.noisy_len)
        for i, r in enumerate(reads[:self.max_cluster_size]):
            oh = one_hot_encode(r, self.noisy_len)
            cluster_left[i]  = oh
            cluster_right[i] = torch.flip(oh, dims=[-1])  # Siamese right branch
        return cluster_left, cluster_right, label_oh

    def get_cluster_size(self, idx): return len(self.clusters[idx])

print("Dataset class defined.")


Dataset class defined.


## Load DNAformer Model & Weights

In [5]:
# Build model
dnaformer = DNAFormerNet(dnaformer_config).to(DEVICE)
total_params = sum(p.numel() for p in dnaformer.parameters())
print(f"DNAformer: {total_params:,} parameters")

# Load official weights
checkpoint = torch.load(str(DNAFORMER_WEIGHTS), map_location=DEVICE, weights_only=False)
if 'model_state_dict' in checkpoint:
    state_dict = checkpoint['model_state_dict']
else:
    state_dict = checkpoint

# Load -- use strict=False in case of minor key mismatches
missing, unexpected = dnaformer.load_state_dict(state_dict, strict=False)
print(f"Loaded weights from {DNAFORMER_WEIGHTS.name}")
if missing:    print(f"  Missing keys: {len(missing)}")
if unexpected: print(f"  Unexpected keys: {len(unexpected)}")
if not missing and not unexpected: print(f"  All keys matched perfectly.")

dnaformer.eval()

# Load test data
test_ds = DNAformerDataset(EVAL_FILE, MAX_CLUSTER_SIZE, NOISY_LEN, LABEL_SEQ_LEN)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)


/homes/shubham/anaconda3/envs/pytorchenv/lib/python3.10/site-packages/torch/nn/modules/transformer.py:282: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


DNAformer: 103,418,588 parameters
Loaded weights from DNAFormer_siamese.pth
  Missing keys: 2
  Unexpected keys: 2
Loaded 109976 clusters from ../Data/BinnedNanoporeTwoFlowcells.txt


## Experiment: DNAformer Gradient Attribution

Same methodology as DNA-GRU: compute gradient of output logits w.r.t. the features
entering the Transformer encoder (i.e., the output of the embedding module).

In [6]:
NUM_GRAD_SAMPLES = min(200, len(test_ds))
grad_indices = random.sample(range(len(test_ds)), NUM_GRAD_SAMPLES)
target_positions = list(range(0, LABEL_SEQ_LEN, max(1, LABEL_SEQ_LEN // 16)))

influence_matrix = np.zeros((LABEL_SEQ_LEN, LABEL_SEQ_LEN), dtype=np.float64)

# Build separate model for gradient analysis
grad_dnaformer = DNAFormerNet(dnaformer_config).to(DEVICE)
if 'model_state_dict' in checkpoint:
    grad_dnaformer.load_state_dict(checkpoint['model_state_dict'], strict=False)
else:
    grad_dnaformer.load_state_dict(checkpoint, strict=False)
grad_dnaformer.eval()

# Transformer encoder needs train mode for backward
grad_dnaformer.encoder.train()
for p in grad_dnaformer.parameters():
    p.requires_grad_(False)

print(f"DNAformer gradient attribution: {NUM_GRAD_SAMPLES} samples x {len(target_positions)} positions")
print(f"Gradient measured at: embedding module output -> transformer input (label-space, L={LABEL_SEQ_LEN})")
print(f"This matches the DNA-GRU measurement point (embedding_module output -> BiGRU input)")

for si in tqdm(grad_indices, desc="DNAformer Gradient"):
    cluster_left, cluster_right, label_oh = test_ds[si]
    cl = cluster_left.unsqueeze(0).to(DEVICE)   # (1, K, 4, T)
    cr = cluster_right.unsqueeze(0).to(DEVICE)   # (1, K, 4, T)

    # ── Forward through alignment + NCI + embedding (no grad needed here) ──
    with torch.no_grad():
        # Alignment (per-read)
        aligned_l = grad_dnaformer.alignement(cl)  # (1, K, 128, T)
        aligned_r = grad_dnaformer.alignement(cr)  # (1, K, 128, T)
        aligned_cat = torch.cat([aligned_l, aligned_r], dim=0)  # (2, K, 128, T)

        # Embedding module includes NCI (sum over cluster dim) + conv + linear
        # Output shape: (2, d_model, L) -- already in label-space
        emb_out = grad_dnaformer.embedding(aligned_cat)  # (2, 1024, 128)

    # ── Detach and require grad HERE: at Transformer input (label-space) ──
    # This is the SAME measurement point as DNA-GRU's feats (embedding_module output)
    emb_out_d = emb_out.detach().requires_grad_(True)  # (2, 1024, L)

    # Transformer
    x = grad_dnaformer.encoder(rearrange(emb_out_d, 'b emb seq -> seq b emb'))

    # Output module
    x = grad_dnaformer.output_module(rearrange(x, 'seq b emb -> b emb seq'))

    # Fusion (combines left and right branches)
    x_fused, _, _ = grad_dnaformer.fusion(x)  # (1, 4, L)

    for tp in target_positions:
        if emb_out_d.grad is not None:
            emb_out_d.grad.zero_()
        x_fused[0, :, tp].max().backward(retain_graph=True)

        if emb_out_d.grad is not None:
            # emb_out_d shape: (2, d_model, L) -- take left branch (index 0)
            # Norm over d_model dimension to get per-position influence in label-space
            gn = emb_out_d.grad[0].norm(dim=0).detach().cpu().numpy()  # shape: (L,)
            # No interpolation needed -- already in label-space!
            s = gn.sum()
            if s > 0:
                influence_matrix[tp, :] += gn / s

    del emb_out_d, x, x_fused
    torch.cuda.empty_cache()

del grad_dnaformer
torch.cuda.empty_cache()

# Normalize
for tp in target_positions:
    if NUM_GRAD_SAMPLES > 0:
        influence_matrix[tp, :] /= NUM_GRAD_SAMPLES

# Compute context radii
context_radii = {50: [], 75: [], 90: [], 95: [], 99: []}
for tp in target_positions:
    row = influence_matrix[tp, :]
    s = row.sum()
    if s == 0: continue
    rn = row / s
    for pct, lst in context_radii.items():
        for w in range(1, LABEL_SEQ_LEN):
            if rn[max(0,tp-w):min(LABEL_SEQ_LEN,tp+w+1)].sum() >= pct/100:
                lst.append(w); break
        else: lst.append(LABEL_SEQ_LEN)

print("\nDNAformer Effective Context Radius (measured at Transformer input):")
cr_summary = {}
for pct, radii in context_radii.items():
    if not radii: continue
    m, md_val, mx = np.mean(radii), np.median(radii), np.max(radii)
    print(f"  {pct}% influence: mean={m:.1f}, median={md_val:.1f}, max={mx}")
    cr_summary[f"pct{pct}"] = {"mean": float(m), "median": float(md_val), "max": int(mx)}

# Save with 'dnaformer_' prefix
np.save(T_RESULTS / f"dnaformer_influence_matrix_{DATASET_NAME}.npy", influence_matrix)
with open(T_RESULTS / f"dnaformer_context_radius_{DATASET_NAME}.json", 'w') as f:
    json.dump(cr_summary, f, indent=2)
sio.savemat(str(T_MAT / f'dnaformer_context_radius_{DATASET_NAME}.mat'), {
    'dataset_name': DATASET_NAME, 'model_name': 'DNAformer',
    'measurement_point': 'embedding_output_to_transformer_input',
    'influence_matrix': influence_matrix,
    'target_positions': np.array(target_positions, dtype=np.float64),
    'label_seq_len': np.float64(LABEL_SEQ_LEN),
    **{f'radius_{p}pct_values': np.array(v, dtype=np.float64) for p, v in context_radii.items() if v},
})
print(f"\n-> Saved with 'dnaformer_' prefix to {T_RESULTS} and {T_MAT}")

DNAformer gradient attribution: 200 samples x 16 positions
Gradient measured at: embedding module output -> transformer input (label-space, L=128)
This matches the DNA-GRU measurement point (embedding_module output -> BiGRU input)


DNAformer Gradient:   0%|          | 0/200 [00:00<?, ?it/s]


DNAformer Effective Context Radius (measured at Transformer input):
  50% influence: mean=33.4, median=29.0, max=64
  75% influence: mean=61.6, median=55.0, max=102
  90% influence: mean=82.0, median=80.0, max=121
  95% influence: mean=89.0, median=88.5, max=125
  99% influence: mean=94.6, median=94.5, max=127

-> Saved with 'dnaformer_' prefix to Experiments/BinnedNanoporeTwoFlowcells_v4/TheoryAnalysis/Results and Experiments/BinnedNanoporeTwoFlowcells_v4/TheoryAnalysis/MatFiles


## Comparison: DNA-GRU vs DNAformer Context Radius

In [7]:
# Load DNA-GRU results for comparison
gru_cr_file = T_RESULTS / "context_radius.json"
if not gru_cr_file.exists():
    gru_cr_file = T_RESULTS / f"context_radius_{DATASET_NAME.lower()}.json"

if gru_cr_file.exists():
    with open(gru_cr_file) as f:
        gru_cr = json.load(f)
    print(f"{'Metric':<20} {'DNA-GRU':>10} {'DNAformer':>10}")
    print("-" * 42)
    for pct in [50, 75, 90, 95, 99]:
        gru_val  = gru_cr.get(f"pct{pct}", {}).get("mean", "N/A")
        df_val   = cr_summary.get(f"pct{pct}", {}).get("mean", "N/A")
        gru_s = f"{gru_val:.1f}" if isinstance(gru_val, float) else str(gru_val)
        df_s  = f"{df_val:.1f}" if isinstance(df_val, float) else str(df_val)
        print(f"  w_{pct:2d}              {gru_s:>10} {df_s:>10}")
else:
    print(f"DNA-GRU results not found at {gru_cr_file}")
    print("Run the DNA-GRU analysis notebook first for comparison.")
    print(f"\nDNAformer-only results:")
    for pct, data in cr_summary.items():
        print(f"  {pct}: mean={data['mean']:.1f}")


DNA-GRU results not found at Experiments/BinnedNanoporeTwoFlowcells_v4/TheoryAnalysis/Results/context_radius_binnednanoporetwoflowcells.json
Run the DNA-GRU analysis notebook first for comparison.

DNAformer-only results:
  pct50: mean=33.4
  pct75: mean=61.6
  pct90: mean=82.0
  pct95: mean=89.0
  pct99: mean=94.6


## Summary

In [8]:
summary = {
    "dataset": DATASET_NAME,
    "model": "DNAformer",
    "total_params": total_params,
    "total_clusters": len(test_ds),
    "context_radius": cr_summary,
}
with open(T_RESULTS / f"dnaformer_summary_{DATASET_NAME}.json", 'w') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print(f"\nAll DNAformer results saved to {THEORY_DIR}")


{
  "dataset": "BinnedNanoporeTwoFlowcells",
  "model": "DNAformer",
  "total_params": 103418588,
  "total_clusters": 109976,
  "context_radius": {
    "pct50": {
      "mean": 33.4375,
      "median": 29.0,
      "max": 64
    },
    "pct75": {
      "mean": 61.625,
      "median": 55.0,
      "max": 102
    },
    "pct90": {
      "mean": 82.0,
      "median": 80.0,
      "max": 121
    },
    "pct95": {
      "mean": 89.0,
      "median": 88.5,
      "max": 125
    },
    "pct99": {
      "mean": 94.5625,
      "median": 94.5,
      "max": 127
    }
  }
}

All DNAformer results saved to Experiments/BinnedNanoporeTwoFlowcells_v4/TheoryAnalysis
